###Ingest BOU_transactions csv file   
1.read file using dataframe reader api   
2.add metadata columns - source file,ingestion timestamp     
3.write bronze tables using dataframe writer api

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment_config


In [0]:
%run ../00-common/02.bronze_functions

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/BOU_transactions/"

In [0]:
table_name = f"{catalog_name}.{bronze_schema}.BOU_transactions"

In [0]:
from pyspark.sql.types import *

In [0]:
BOU_schema = StructType([
  StructField('TxnID', StringType()),
  StructField('BillerID', StringType()),
  StructField('CustomerID', StringType()),
  StructField('TxnAmount', DoubleType()),
  StructField('Status', StringType()),
  StructField('BillerRefID', StringType()),
  StructField('Timestamp', TimestampType())
])

In [0]:
BOU_df = spark.read.format('csv') \
                .option('header', True) \
                .schema(BOU_schema) \
                .option('mode', 'FAILFAST') \
                .load(source_file)

In [0]:
BOU_audit = add_ingestion_metadata(BOU_df)

In [0]:
BOU_final= BOU_audit.withColumn("batch_id", F.lit(v_batch_id))

In [0]:
BOU_final.write.mode('overwrite').partitionBy('batch_id').option('replaceWhere',f"batch_id = '{v_batch_id}'").saveAsTable(table_name)